# EEE355 — Computation Structures
## Week 1: Introduction to Computation and Digital Abstraction

**Instructor:** Your Name  
**Session:** 2026/2027  
**Class:** EEE355  
**Lesson Type:** Teacher-Prepared Interactive Class Note

## Teacher Preparation Instructions

Before sharing this notebook with students, edit the configuration cell below and fill in:

- `API_BASE`
- `ASSESSMENT_ID`
- `ATTENDANCE_SESSION_ID`
- `LESSON_SLUG`
- `QUESTION_1_ID`
- any additional question IDs you need

Students will log in inside the notebook using a small login widget, so they do **not** need to paste a token manually.

In [ ]:
# ===== Teacher Prepared Classroom Configuration =====
API_BASE = "http://localhost:8000"

# Teacher fills these before class
ASSESSMENT_ID = 1
ATTENDANCE_SESSION_ID = 1
LESSON_SLUG = "week01_intro"
QUESTION_1_ID = 1

# Optional extra question IDs
QUESTION_2_ID = None
QUESTION_3_ID = None

## Student Login

Students should enter their email and password below, then click **Login**.

The password field is hidden while typing.

In [ ]:
import json
import requests
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

STUDENT_TOKEN = None
CURRENT_USER = None

email_input = widgets.Text(
    description="Email:",
    placeholder="student@example.com",
    layout=widgets.Layout(width="420px")
)

password_input = widgets.Password(
    description="Password:",
    placeholder="Enter password",
    layout=widgets.Layout(width="420px")
)

login_button = widgets.Button(
    description="Login",
    button_style="primary"
)

login_status = widgets.Output()

def login_student(_):
    global STUDENT_TOKEN, CURRENT_USER
    with login_status:
        clear_output()
        try:
            response = requests.post(
                f"{API_BASE}/api/auth/login",
                json={
                    "email": email_input.value.strip(),
                    "password": password_input.value
                },
                timeout=30
            )
            response.raise_for_status()
            data = response.json()
            STUDENT_TOKEN = data["access_token"]

            me_response = requests.get(
                f"{API_BASE}/api/auth/me",
                headers={"Authorization": f"Bearer {STUDENT_TOKEN}"},
                timeout=30
            )
            me_response.raise_for_status()
            CURRENT_USER = me_response.json()

            print(f"Logged in successfully as {CURRENT_USER['full_name']} ({CURRENT_USER['email']})")
        except Exception as exc:
            STUDENT_TOKEN = None
            CURRENT_USER = None
            print(f"Login failed: {exc}")

login_button.on_click(login_student)

display(widgets.VBox([email_input, password_input, login_button, login_status]))

In [ ]:
HEADERS = {}

student_answers = {}
attendance_record = {}
attempt_info = None
paper = None

def require_token():
    global HEADERS
    if not STUDENT_TOKEN:
        raise ValueError("Please log in first using the login widget.")
    HEADERS = {
        "Authorization": f"Bearer {STUDENT_TOKEN}",
        "Content-Type": "application/json",
    }

## Lesson Overview

In this lesson we will cover:

- what computation means
- information representation
- digital abstraction
- why binary systems are useful in digital engineering

## Main Note Content

### What is Computation?

Computation is the process of transforming information according to rules.

In engineering, computation may involve:

- arithmetic processing
- logical decision making
- signal representation
- system control

A computing system takes **input**, performs **processing**, and produces **output**.

## Digital Abstraction

Digital abstraction means representing information using discrete values instead of continuous ones.

For example:

- a light may be represented as **ON/OFF**
- a switch may be represented as **1/0**
- a truth value may be represented as **TRUE/FALSE**

This makes design, storage, and processing more reliable.

In [ ]:
def analog_to_digital_example(value, threshold=0.5):
    return 1 if value >= threshold else 0

samples = [0.1, 0.3, 0.49, 0.5, 0.7, 0.95]
converted = [(x, analog_to_digital_example(x)) for x in samples]
converted

## Start the Mock Exam Attempt

Run this cell once to start or resume the backend attempt for this lesson's quick check.

In [ ]:
require_token()

response = requests.post(
    f"{API_BASE}/api/mock-exams/{ASSESSMENT_ID}/start",
    headers=HEADERS,
    timeout=30,
)
response.raise_for_status()
attempt_info = response.json()
attempt_info

In [ ]:
require_token()

response = requests.get(
    f"{API_BASE}/api/mock-exams/{ASSESSMENT_ID}/paper",
    headers=HEADERS,
    timeout=30,
)
response.raise_for_status()
paper = response.json()
paper

## Quick Check

Answer the following questions before moving on.

In [ ]:
def answer_mcq(question_id, selected_option):
    student_answers[question_id] = {"selected_option": selected_option}
    return {"saved": question_id, "response": student_answers[question_id]}

def answer_multi(question_id, selected_options):
    student_answers[question_id] = {"selected_options": selected_options}
    return {"saved": question_id, "response": student_answers[question_id]}

def answer_fill_gap(question_id, gaps):
    student_answers[question_id] = {"gaps": gaps}
    return {"saved": question_id, "response": student_answers[question_id]}

def answer_essay(question_id, answer_text):
    student_answers[question_id] = {"answer_text": answer_text}
    return {"saved": question_id, "response": student_answers[question_id]}

In [ ]:
question_1 = '''
Q1. Which of the following best describes digital abstraction?

a. Representation using continuous values only
b. Representation using discrete values
c. Elimination of all physical signals
d. Use of decimal numbers only
'''
print(question_1)

In [ ]:
# Teacher already filled QUESTION_1_ID above.
# Student only changes the selected option.
answer_mcq(question_id=QUESTION_1_ID, selected_option="b")
student_answers

## Autosave Answers to Backend

Run this cell after answering questions.

In [ ]:
require_token()

if attempt_info is None:
    raise ValueError("Start the mock exam attempt first.")

payload = {
    "responses": [
        {"question_id": qid, "response": resp}
        for qid, resp in student_answers.items()
    ]
}

response = requests.post(
    f"{API_BASE}/api/mock-exams/attempts/{attempt_info['attempt_id']}/autosave",
    headers=HEADERS,
    data=json.dumps(payload),
    timeout=30,
)
response.raise_for_status()
response.json()

## Attendance Check

Complete the attendance confirmation below during class time.

In [ ]:
attendance_record = {
    "attendance_session_id": ATTENDANCE_SESSION_ID,
    "status": "present",
}
attendance_record

In [ ]:
require_token()

response = requests.post(
    f"{API_BASE}/api/courses/attendance/mark",
    headers=HEADERS,
    data=json.dumps(attendance_record),
    timeout=30,
)
response.raise_for_status()
response.json()

## Submit Today’s Work

Run this after you have answered all quick-check questions and marked attendance.

In [ ]:
require_token()

if attempt_info is None:
    raise ValueError("Start the mock exam attempt first.")

submitted_payload = {
    "lesson": LESSON_SLUG,
    "submitted_from": "jupyterlite",
    "submitted_at": datetime.utcnow().isoformat(),
    "attendance_session_id": ATTENDANCE_SESSION_ID,
    "done": True,
}

response = requests.post(
    f"{API_BASE}/api/mock-exams/attempts/{attempt_info['attempt_id']}/submit",
    headers=HEADERS,
    data=json.dumps({"submitted_payload": submitted_payload}),
    timeout=30,
)
response.raise_for_status()
response.json()

## Check Score Summary

Objective and fill-gap items may be scored immediately. Essay/theory items may wait for teacher review.

In [ ]:
require_token()

if attempt_info is None:
    raise ValueError("Start the mock exam attempt first.")

response = requests.get(
    f"{API_BASE}/api/mock-exams/attempts/{attempt_info['attempt_id']}/scores",
    headers=HEADERS,
    timeout=30,
)
response.raise_for_status()
response.json()

## Assignment / Next Steps

1. Review today’s note.
2. Explain digital abstraction in your own words.
3. Give two examples of discrete representation in engineering systems.
4. Prepare for next class on number systems and binary representation.